# Max-plus eigenstructure playground

This notebook is a dataset-agnostic example. Replace the synthetic CSV with your own adjacency matrix or edge list; the analysis code remains unchanged.


In [ ]:
import Pkg

repo_root = if isfile(joinpath(pwd(), "Project.toml"))
    pwd()
elseif isfile(joinpath(pwd(), "..", "Project.toml"))
    normpath(joinpath(pwd(), ".."))
else
    error("Run this notebook from the repository root or notebooks/ directory.")
end

Pkg.activate(repo_root)
using Maxplus_eigenvectors

## Load data

The package convention is `A[target, source]`. For dense adjacency matrices, numeric zero means an absent edge by default.

In [ ]:
input_file = joinpath(repo_root, "data", "examples", "weighted_network.csv")
data = load_adjacency_csv(input_file)
A = data.matrix
labels = data.labels

network_summary((example=A,))

An edge-list CSV can be loaded instead with `load_edgelist_csv(path)`. Any square Julia matrix can also be passed directly to `analyse_network`.

## Max-plus eigendecomposition

In [ ]:
analysis = analyse_network(
    A;
    labels=labels,
    name="synthetic example",
)

In [ ]:
spectrum = finite_right_spectrum_summary(analysis.right)
supports = finite_right_eigenvector_support(analysis.right, labels)

display(spectrum)
display(supports)

## Critical structure and SCCs

In [ ]:
if size(spectrum, 1) > 0
    block = spectrum.Block[1]
    direction = spectrum.Direction[1]

    cs = critical_structure(
        analysis.matrix,
        analysis.right;
        block=block,
        direction=direction,
    )
    display(critical_structure_summary(cs, labels))

    scc = analyse_scc_structure(
        analysis.matrix,
        analysis.right,
        labels;
        block=block,
        direction=direction,
    )
    display(scc.summary)
end

## Optional Graphviz output

If Graphviz is installed, uncomment the calls below. Generated files are written to `notebooks/figs/` and ignored by Git.

In [ ]:
# visualize_critical_structure(
#     analysis.matrix, analysis.right, labels;
#     block=block, direction=direction,
#     filename=joinpath(repo_root, "notebooks", "figs", "critical_structure"),
# )
#
# visualize_scc_structure(
#     analysis.matrix, analysis.right, labels;
#     block=block, direction=direction,
#     filename=joinpath(repo_root, "notebooks", "figs", "scc_structure"),
# )